# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).


### 1. Cài đặt thư viện cần thiết

In [13]:
# Sử dụng pip để cài đặt các thư viện, cờ -q để giảm output
!pip install -q fastapi uvicorn python-dotenv chromadb pydantic torch nest_asyncio


### 2. Import thư viện và Cấu hình

In [14]:
import os
import json
import requests
import torch
import torch.nn.functional as F
from typing import List, Optional, Dict
from chromadb.api.types import Documents, Embeddings
from chromadb.utils.embedding_functions import EmbeddingFunction
from dotenv import load_dotenv
import chromadb
import numpy as np
from pydantic import BaseModel
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
import nest_asyncio

# Apply the patch to allow nested event loops in Jupyter
nest_asyncio.apply()

# Load biến từ file .env vào môi trường
load_dotenv()

class Settings:
    API_KEY = os.getenv("FPT_API_KEY")
    ENDPOINT = os.getenv("FPT_ENDPOINT")
    EMBED_MODEL = "Vietnamese_Embedding"
    L_LLM_MODEL = "Llama-3.3-Swallow-70B-Instruct-v0.4"
    H_LLM_MODEL = "DeepSeek-R1"
    DB_PATH = "./chroma_data"
    COLLECTION_NAME = "job_listings"

# Tạo một biến settings để dùng chung
settings = Settings()

print("✅ Đã load cấu hình và các thư viện (đã áp dụng nest_asyncio).")


✅ Đã load cấu hình và các thư viện (đã áp dụng nest_asyncio).


### 3. Khai báo các Client và Adapter
<p>Đây là nơi định nghĩa các lớp giao tiếp với API của FPT và VectorDB Chroma.</p>

In [40]:
class FPTAIClient:
    def __init__(self):
        self.headers = {
            "Authorization": f"Bearer {settings.API_KEY}",
            "Content-Type": "application/json"
        }
        self.endpoint = settings.ENDPOINT.rstrip('/')

    def get_embedding(self, text: str) -> List[float]:
        url = f"{self.endpoint}/embeddings"
        payload = {"input": [text], "model": settings.EMBED_MODEL}
        try:
            response = requests.post(url, headers=self.headers, json=payload)
            response.raise_for_status()
            data = response.json()
            if "data" in data and len(data["data"]) > 0:
                return data["data"][0]["embedding"]
            return []
        except Exception as e:
            print(f"Lỗi Embedding: {e}")
            return []

    def chat_refine(self, text: str) -> str:
        url = f"{self.endpoint}/chat/completions"
        prompt = f"{text}\n\nHãy tóm tắt lại thông tin ứng viên ngắn gọn."
        payload = {
            "model": settings.L_LLM_MODEL,
            "messages": [
                {"role": "system", "content": "Bạn là trợ lý HR."},
                {"role": "user", "content": prompt}
            ],
            "temperature": 0.1
        }
        try:
            response = requests.post(url, headers=self.headers, json=payload)
            response.raise_for_status()
            return response.json()["choices"][0]["message"]["content"]
        except Exception as e:
            print(f"Lỗi Chat: {e}")
            return "Lỗi xử lý văn bản"

    def generate_report(self, job_description: str, cv_text: str) -> str:
        url = f"{self.endpoint}/chat/completions"
        prompt = f"""
        Dưới đây là hồ sơ ứng viên:
        {cv_text}

        Dưới đây là mô tả công việc:
        {job_description}

        Hãy phân tích:
        - Những điểm phù hợp chính.
        - Những điểm còn thiếu và có thể đào tạo nhanh.
        - Tóm tắt mức độ phù hợp dạng phần trăm.
        - Gợi ý công ty nên trao đổi gì khi phỏng vấn.
        - Gợi ý ứng viên nên học thêm gì nếu muốn tăng cơ hội trúng tuyển.
        """
        payload = {
            "model": settings.H_LLM_MODEL,
            "messages": [
                {"role": "system", "content": "Bạn là chuyên gia phân tích việc làm của Jobcubator. Nhiệm vụ: đánh giá mức độ phù hợp giữa ứng viên và công việc, dựa trên thông tin đã cho. Chỉ đưa ra câu trả lời dựa trên dữ liệu đầu vào. Không tự bịa thêm thông tin."},
                {"role": "user", "content": prompt}
            ],
            "temperature": 0.2
        }
        try:
            response = requests.post(url, headers=self.headers, json=payload)
            response.raise_for_status()
            return response.json()["choices"][0]["message"]["content"]
        except Exception as e:
            print(f"Lỗi Chat: {e}")
            return "Lỗi xử lý văn bản"

    def chat_respond_custom(self, user_text: str, sys_prompt: str):
        url = f"{self.endpoint}/chat/completions"
        payload = {
            "model": settings.H_LLM_MODEL,
            "messages": [
                {"role": "system", "content": sys_prompt},
                {"role": "user", "content": user_text}
            ],
            "temperature": 0.3,
            "stream": True
        }
        try:
            response = requests.post(url, headers=self.headers, json=payload, stream=True)
            response.raise_for_status()
            for line in response.iter_lines():
                if not line: continue
                text = line.decode("utf-8")
                if text.startswith("data: "): text = text[6:].strip()
                if text == "[DONE]": break
                yield json.loads(text)
        except Exception as e:
            print(f"Lỗi Chat Stream: {e}")
            yield {"error": f"Lỗi xử lý văn bản: {e}"}

    def normalize_question(self, text: str) -> str:
        url = f"{self.endpoint}/chat/completions"
        payload = {
            "model": settings.L_LLM_MODEL,
            "messages": [
                {"role": "system", "content": "Nhiệm vụ: Viết lại câu hỏi của người dùng ngắn gọn, rõ ràng, đủ chủ ngữ vị ngữ để AI dễ hiểu. Không trả lời, chỉ viết lại."},
                {"role": "user", "content": text}
            ],
            "temperature": 0.1
        }
        try:
            res = requests.post(url, headers=self.headers, json=payload)
            res.raise_for_status()
            return res.json()["choices"][0]["message"]["content"]
        except Exception as e:
            print(f"⚠️ Lỗi khi chuẩn hóa câu hỏi: {e}. Sử dụng câu hỏi gốc.")
            return text

    def smart_chat(self, user_text: str, router_instance):
        instruction, is_match = router_instance.find_best_instruction(user_text)

        if is_match:
            print("🎯 HIT: Trúng câu hỏi mẫu -> Dùng Instruction chuyên gia.")
            # For the HIT case, we combine the expert instruction directly into the user prompt
            # for maximum effect. This creates a stronger signal for the LLM.
            system_prompt = "Bạn là một trợ lý AI chuyên nghiệp, luôn trả lời bằng tiếng Việt."
            final_prompt = f"""Dựa trên vai trò của bạn là một chuyên gia, hãy trả lời câu hỏi sau một cách chi tiết.

**Hướng dẫn vai trò (Instruction):**
{instruction}

**Câu hỏi của người dùng:**
{user_text}
"""
        else:
            print("⚠️ MISS: Câu hỏi lạ -> Dùng Light LLM chuẩn hóa.")
            # For the MISS case, we stick to the original strategy of refining the question
            # and using a standard system prompt.
            system_prompt = "Bạn là trợ lý HR hữu ích, luôn trả lời bằng tiếng Việt."
            refined_text = self.normalize_question(user_text)
            print(f"   Câu hỏi đã chuẩn hóa: {refined_text}")
            final_prompt = refined_text

        return self.chat_respond_custom(final_prompt, system_prompt)

class FPTChromaAdapter(EmbeddingFunction):
    def __init__(self, ai_client: FPTAIClient):
        self.ai_client = ai_client

    def __call__(self, texts: Documents) -> Embeddings:
        embeddings = []
        for text in texts:
            embedding = self.ai_client.get_embedding(text)
            embeddings.append(embedding)
        return embeddings

class VectorDBClient:
    def __init__(self, ai_client: FPTAIClient):
        print(f"📦 Đang khởi tạo VectorDB với model: {settings.EMBED_MODEL}")
        self.client = chromadb.PersistentClient(path=settings.DB_PATH)
        self.embedding_func = FPTChromaAdapter(ai_client=ai_client)
        self.collection = self.client.get_or_create_collection(
            name=settings.COLLECTION_NAME,
            embedding_function=self.embedding_func
        )

    def add_jobs(self, jobs_data: list):
        if not jobs_data: return False
        try:
            ids = [str(j["id"]) for j in jobs_data]
            documents = [j["description"] for j in jobs_data]
            metadatas = [{"category": j.get("category", "General"), "title": j.get("title", "N/A")} for j in jobs_data]
            self.collection.add(ids=ids, documents=documents, metadatas=metadatas)
            print(f"✅ Đã thêm {len(ids)} jobs vào DB.")
            return True
        except Exception as e:
            print(f"❌ Lỗi thêm job: {e}")
            return False

    def search_similar_jobs(self, query_text: str, n_results=5):
        try:
            results = self.collection.query(query_texts=[query_text], n_results=n_results)
            clean_results = []
            if results and results['documents']:
                for i in range(len(results['documents'][0])):
                    clean_results.append({
                        "id": results['ids'][0][i],
                        "description": results['documents'][0][i],
                        "metadata": results['metadatas'][0][i] if results['metadatas'] else {}
                    })
            return clean_results
        except Exception as e:
            print(f"❌ Lỗi tìm kiếm Vector: {e}")
            return []

print("✅ Đã khai báo các Client và Adapter.")


✅ Đã khai báo các Client và Adapter.


### 4. Khai báo các lớp xử lý logic (Router, Matcher)

In [41]:
class SemanticRouter:
    def __init__(self, fpt_client, intent_file="respond.json"):
        self.client = fpt_client
        if os.path.exists(intent_file):
            with open(intent_file, "r", encoding="utf-8") as f:
                self.intents = json.load(f)
        else:
            print(f"❌ Lỗi: Không tìm thấy file '{intent_file}'")
            self.intents = []

        self.sample_vectors = []
        self.intent_map = []
        print("🔄 Đang khởi tạo Router...")
        for item in self.intents:
            for sample in item["samples"]:
                vec_list = self.client.get_embedding(sample)
                if vec_list:
                    vec_tensor = torch.tensor(vec_list, dtype=torch.float32)
                    self.sample_vectors.append(vec_tensor)
                    self.intent_map.append(item["system_instruction"])
        
        if self.sample_vectors:
            self.sample_vectors = torch.stack(self.sample_vectors)
            print("✅ Router đã sẵn sàng!")
        else:
            print("⚠️ Cảnh báo: Không tạo được vector nào cho Router.")

    def find_best_instruction(self, user_query: str, threshold=0.65): # GIẢM NGƯỠNG XUỐNG 0.65
        # Sửa lỗi: Kiểm tra một cách tường minh, không dùng `if not tensor`
        if not isinstance(self.sample_vectors, torch.Tensor) or self.sample_vectors.numel() == 0:
            return None, False
        
        query_list = self.client.get_embedding(user_query)
        if not query_list: return None, False
        
        query_vec = torch.tensor(query_list, dtype=torch.float32)
        scores = F.cosine_similarity(query_vec, self.sample_vectors)
        max_score, idx = torch.max(scores, dim=0)
        print(f"🔍 Router Score: {max_score.item():.2f}")
        
        if max_score.item() >= threshold:
            return self.intent_map[idx.item()], True
        return None, False

    def get_all_suggestions(self):
        suggestion_list = []
        for category in self.intents:
            suggestion_list.extend(category["samples"][:2])
        return suggestion_list

class JobMatcher:
    @staticmethod
    def compute_similarity(vec1: List[float], vec2: List[float]) -> float:
        if not vec1 or not vec2: return 0.0
        t1 = torch.tensor(vec1)
        t2 = torch.tensor(vec2)
        return F.cosine_similarity(t1, t2, dim=0).item()

print("✅ Đã khai báo Router và Matcher.")


✅ Đã khai báo Router và Matcher.


### 5. Khởi tạo và định nghĩa API Endpoints với FastAPI

In [42]:
# Khai báo Schema Pydantic cho API
class TextRequest(BaseModel):
    text: str
    user_id: Optional[str] = None

class MatchRequest(BaseModel):
    user_embedding: List[float]
    job_embedding: List[float]

class PipelineRequest(BaseModel):
    cv_text: str

# Khởi tạo các đối tượng chính
ai_client = FPTAIClient()
db_client = VectorDBClient(ai_client=ai_client)
intent_router = SemanticRouter(fpt_client=ai_client)
matcher = JobMatcher()

# Khởi tạo App FastAPI
app = FastAPI()

@app.get("/")
def root():
    return {"message": "Jobcubator API is running!"}

@app.post("/embed")
def endpoint_embed(data: TextRequest):
    vector = ai_client.get_embedding(data.text)
    if not vector:
        raise HTTPException(status_code=500, detail="Lỗi tạo embedding từ AI")
    return {"embedding": vector}

@app.post("/pipeline/find_jobs")
def endpoint_pipeline(data: PipelineRequest):
    # 1. Tóm tắt CV để tìm kiếm tốt hơn
    refined_text = ai_client.chat_refine(data.cv_text)

    # 2. Dùng CV đã tóm tắt để tìm các job tương tự trong DB
    similar_jobs = db_client.search_similar_jobs(refined_text, n_results=3)
    if not similar_jobs:
        return {"message": "Không tìm thấy công việc phù hợp.", "original_cv": data.cv_text, "refined_cv": refined_text, "results": []}

    # 3. (Tùy chọn) Tạo báo cáo phân tích cho job phù hợp nhất
    best_job = similar_jobs[0]
    report = ai_client.generate_report(job_description=best_job['description'], cv_text=data.cv_text)
    
    return {
        "original_cv": data.cv_text,
        "refined_cv": refined_text,
        "report_for_best_match": report,
        "similar_jobs": similar_jobs
    }

@app.post("/chat/general")
def endpoint_general_chat(data: TextRequest):
    def output_generator():
        stream = ai_client.smart_chat(data.text, intent_router)
        for chunk in stream:
            if "choices" in chunk and len(chunk["choices"]) > 0:
                delta = chunk["choices"][0].get("delta", {})
                if "content" in delta:
                    yield delta["content"]
    return StreamingResponse(output_generator(), media_type="text/plain")

@app.get("/config/suggestions")
def get_chat_suggestions():
    suggestions = intent_router.get_all_suggestions()
    return {"suggestions": suggestions, "count": len(suggestions)}

print("✅ Đã khởi tạo FastAPI app và các endpoints.")


📦 Đang khởi tạo VectorDB với model: Vietnamese_Embedding
🔄 Đang khởi tạo Router...
✅ Router đã sẵn sàng!
✅ Đã khởi tạo FastAPI app và các endpoints.


### 6. (QUAN TRỌNG) Thêm dữ liệu mẫu vào VectorDB
<p>Bạn cần chạy cell này ít nhất một lần để có dữ liệu cho việc tìm kiếm. Nếu không, kết quả tìm kiếm sẽ luôn rỗng.</p>

In [43]:
# Tạo file respond.json nếu chưa có
respond_json_content = """
[
  {
    "intent": "Hỏi về tư vấn nghề nghiệp",
    "system_instruction": "Bạn là chuyên gia tư vấn hướng nghiệp. Hãy trả lời các câu hỏi của người dùng về định hướng công việc, kỹ năng cần thiết và lộ trình phát triển.",
    "samples": [
      "Tôi nên học gì để làm trong ngành AI?",
      "Làm sao để trở thành một data scientist?",
      "Ngành IT giờ có bão hòa không?",
      "Em muốn chuyển ngành thì bắt đầu từ đâu?"
    ]
  },
  {
    "intent": "Hỏi về cách viết CV",
    "system_instruction": "Bạn là chuyên gia tuyển dụng nhân sự. Hãy đưa ra lời khuyên giúp người dùng cải thiện CV của họ để gây ấn tượng với nhà tuyển dụng.",
    "samples": [
      "CV của em cần sửa gì không?",
      "Làm sao để viết CV cho người mới ra trường?",
      "Viết CV cho vị trí senior developer thì cần chú ý gì?"
    ]
  }
]
"""
if not os.path.exists("respond.json"):
    with open("respond.json", "w", encoding="utf-8") as f:
        f.write(respond_json_content)
    print("📝 Đã tạo file respond.json mẫu.")

# Dữ liệu jobs mẫu
sample_jobs = [
    {"id": 1, "title": "Data Scientist", "category": "Data", "description": "Yêu cầu kinh nghiệm làm việc với Python, SQL và các thư viện học máy như Scikit-learn, TensorFlow. Có khả năng xây dựng mô hình dự đoán và phân tích dữ liệu lớn."},
    {"id": 2, "title": "Backend Developer (Python/Django)", "category": "Web", "description": "Tuyển dụng lập trình viên Backend thông thạo Python và framework Django. Kinh nghiệm làm việc với RESTful API, PostgreSQL và Docker là một lợi thế."},
    {"id": 3, "title": "Frontend Developer (ReactJS)", "category": "Web", "description": "Tìm kiếm ứng viên có kinh nghiệm từ 2 năm trở lên với ReactJS, Redux, và HTML/CSS. Có kinh nghiệm làm việc với các UI component library."},
    {"id": 4, "title": "AI Engineer", "category": "AI", "description": "Cần tuyển kỹ sư AI có kiến thức về xử lý ngôn ngữ tự nhiên (NLP) và thị giác máy tính (Computer Vision). Sử dụng thành thạo PyTorch hoặc TensorFlow."},
    {"id": 5, "title": "Kế toán tổng hợp", "category": "Finance", "description": "Yêu cầu tốt nghiệp chuyên ngành kế toán, kiểm toán. Có ít nhất 3 năm kinh nghiệm ở vị trí tương đương, nắm vững các quy định về thuế và báo cáo tài chính."}
]

# Khởi tạo lại client và thêm dữ liệu
ai_client_for_db = FPTAIClient()
db_client_for_db = VectorDBClient(ai_client=ai_client_for_db)
db_client_for_db.add_jobs(sample_jobs)


📦 Đang khởi tạo VectorDB với model: Vietnamese_Embedding
✅ Đã thêm 5 jobs vào DB.


True

### 7. Chạy thử nghiệm API
<p>Bạn có thể dùng các đoạn code dưới đây để test các endpoint đã tạo.</p>

In [44]:
# Test tìm kiếm công việc
# Giả lập một CV đầu vào
my_cv = "Tôi là một lập trình viên có kinh nghiệm 3 năm với Python và các mô hình AI. Tôi đã làm việc với các dự án về xử lý ngôn ngữ tự nhiên."

# Tạo request
request_data = PipelineRequest(cv_text=my_cv)

# Gọi hàm xử lý của endpoint để test
test_result = endpoint_pipeline(request_data)

# In kết quả
import pprint
pprint.pprint(test_result)


{'original_cv': 'Tôi là một lập trình viên có kinh nghiệm 3 năm với Python và '
                'các mô hình AI. Tôi đã làm việc với các dự án về xử lý ngôn '
                'ngữ tự nhiên.',
 'refined_cv': 'Một lập trình viên có 3 năm kinh nghiệm với Python và các mô '
               'hình AI, chuyên về xử lý ngôn ngữ tự nhiên.',
 'report_for_best_match': '<think>\n'
                          "Okay, let's start by analyzing the candidate's "
                          'profile against the job description. The candidate '
                          'has 3 years of experience with Python and AI '
                          'models, specifically in NLP projects. The job '
                          'requires expertise in both NLP and Computer Vision, '
                          'along with proficiency in PyTorch or TensorFlow.\n'
                          '\n'
                          "First, the matches: The candidate's experience with "
                          "NLP and Python is a stron

In [45]:
# Test chat (Phiên bản đồng bộ đơn giản) để thử trường hợp HIT

def run_chat_test_sync(text: str):
    print("="*50)
    print(f"Hỏi: {text}")
    
    # Gọi thẳng vào hàm xử lý logic, bỏ qua lớp API và StreamingResponse
    stream_generator = ai_client.smart_chat(text, intent_router)

    print("Đáp:")
    
    # Vì stream_generator là một generator đồng bộ, ta dùng vòng lặp for bình thường
    for chunk in stream_generator:
        # Lấy nội dung từ chunk JSON trả về
        if chunk and "choices" in chunk and len(chunk["choices"]) > 0:
            delta = chunk["choices"][0].get("delta", {})
            if "content" in delta:
                print(delta["content"], end="")
    print("\n" + "="*50 + "\n")

# --- Chạy kịch bản HIT ---
# Câu hỏi này rất giống với mẫu "Cách đàm phán mức lương khi phỏng vấn" trong respond.json.
# Mong đợi: Router Score cao, log in ra "🎯 HIT", và câu trả lời chi tiết từ chuyên gia đàm phán lương.
print("--- KỊCH BẢN TEST 'HIT' CASE (ĐÀM PHÁN LƯƠNG) ---")
run_chat_test_sync("Làm thế nào để đàm phán lương khi đi phỏng vấn?")



--- KỊCH BẢN TEST 'HIT' CASE (ĐÀM PHÁN LƯƠNG) ---
Hỏi: Làm thế nào để đàm phán lương khi đi phỏng vấn?
🔍 Router Score: 0.93
🎯 HIT: Trúng câu hỏi mẫu -> Dùng Instruction chuyên gia.
Đáp:
<think>
Đầu tiên, mình cần hiểu rõ yêu cầu của người dùng. Họ muốn biết cách đàm phán lương khi đi phỏng vấn, cụ thể là áp dụng chiến thuật "3 con số" và cách trả lời khéo léo để không bị hớ. Với vai trò là chuyên gia đàm phán lương từ Jobcubator, mình phải đưa ra hướng dẫn chi tiết, tự tin và sắc sảo.

Mình nghĩ đến chiến thuật "3 con số" mà các chuyên gia thường nhắc đến. Đó có lẽ là việc chuẩn bị ba mức lương khác nhau: mức tối thiểu, mức mong đợi và mức lý tưởng. Nhưng cần giải thích rõ từng mức này để người đọc hiểu cách áp dụng.

Sau đó, phần trả lời khéo léo khi nhà tuyển dụng hỏi về mức lương mong muốn. Mình cần đưa ra các ví dụ cụ thể về cách diễn đạt, như hỏi lại phạm vi lương của công ty trước, hoặc trình bày mức lương dựa trên nghiên cứu thị trường. Cũng cần nhấn mạnh tầm quan trọng của việc

#### Minh họa so sánh Embedding
<p>Cell này cho thấy cách so sánh độ tương đồng giữa các đoạn văn bản khác nhau.</p>

In [ ]:
# 1. Định nghĩa các câu cần so sánh
text_1 = "Tuyển dụng kỹ sư phần mềm Python"
text_2 = "Tìm kiếm lập trình viên Python"
text_3 = "Báo cáo tài chính quý 4"

# 2. Lấy vector embedding cho từng câu
print("🔄 Đang lấy embedding cho các câu...")
vec_1 = ai_client.get_embedding(text_1)
vec_2 = ai_client.get_embedding(text_2)
vec_3 = ai_client.get_embedding(text_3)
print("✅ Lấy embedding thành công.")

# 3. Tính toán độ tương đồng
# Dùng đối tượng `matcher` đã được tạo ở cell trên
score_1_2 = matcher.compute_similarity(vec_1, vec_2)
score_1_3 = matcher.compute_similarity(vec_1, vec_3)

# 4. In kết quả
print("\n--- KẾT QUẢ SO SÁNH ---")
print(f"Câu 1: '{text_1}'")
print(f"Câu 2: '{text_2}'")
print(f"Câu 3: '{text_3}'")
print("-" * 20)
# Kết quả mong đợi: score_1_2 sẽ cao hơn score_1_3 đáng kể
print(f"Độ tương đồng (Câu 1 vs Câu 2): {score_1_2:.4f}  <-- Cao, vì cùng chủ đề")
print(f"Độ tương đồng (Câu 1 vs Câu 3): {score_1_3:.4f}  <-- Thấp, vì khác chủ đề")



### 8. Chạy ứng dụng FastAPI với Uvicorn
<p>Để chạy ứng dụng thực sự và cho phép các client khác gọi vào, hãy chạy cell này. Bạn có thể truy cập vào địa chỉ http://127.0.0.1:8000/docs để xem giao diện Swagger UI.</p>

In [12]:
import uvicorn
import asyncio

# Để đơn giản, bạn có thể lưu file này thành .py và chạy bằng command line:
# uvicorn ten_file:app --reload

# Hoặc chạy trực tiếp (có thể có vấn đề với event loop của Jupyter)
# config = uvicorn.Config(app, host="127.0.0.1", port=8000)
# server = uvicorn.Server(config)
# await server.serve()

print("Để chạy server, hãy lưu notebook thành file .py và chạy lệnh:")
print("uvicorn ten_file:app --reload")
print("Ví dụ: uvicorn main:app --reload")


Để chạy server, hãy lưu notebook thành file .py và chạy lệnh:
uvicorn ten_file:app --reload
Ví dụ: uvicorn main:app --reload
